In [1]:
platformID = 'TWI'

## Status 
twitter data is currently ingested and the process of adding facebook factors to it has been started, however the output files still show significant differences to minnie's dataset. 
next step here is retesting the input files and then step by step through the combination process. 

twitter business unit and aggregated services is currently calculated by using minnie's dataset (helper/tw_minnie_preBU.csv)

In [2]:
from datetime import datetime
import pandas as pd
import numpy as np
import psycopg2


## import helper

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import execute_sql_query
import test_functions

In [4]:
# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Year'] == gam_info['file_timeinfo']]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == platformID]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
socialmedia_accounts['Channel ID'] = socialmedia_accounts['Channel ID'].dropna().apply(lambda x: str(int(x)))

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

### RUN TESTS
test_functions.test_lookup_files(country_codes, ['PlaceID'], [0,1,2], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c', 'WeekNumber_finYear'], [3,4,5], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [6,7,8], 
                                 test_step = "lookup files - ensuring social media accounts is correct")



Test 0 passed: lookup DataFrame is not empty.
...updating logbook...

Test 1 passed: No combinations occurs more than once a week.
...updating logbook...

Test 2 passed: No missing values in lookup.
...updating logbook...

Test 3 passed: lookup DataFrame is not empty.
...updating logbook...

Test 4 passed: No combinations occurs more than once a week.
...updating logbook...

Test 5 passed: No missing values in lookup.
...updating logbook...

Test 6 passed: lookup DataFrame is not empty.
...updating logbook...

Test 7 failed: The following combinations occur more than once a week
...updating logbook...

Test 8 passed: No missing values in lookup.
...updating logbook...



# ingestion

In [5]:
tw_activity_df_raw = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",
                             keep_default_na=False)

tw_activity_df_raw['Channel ID'] = tw_activity_df_raw['Channel ID'].apply(lambda x: str(int(x)))
tw_activity_df = tw_activity_df_raw.merge(week_tester[['w/c', 'WeekNumber_finYear']], 
                                          on='w/c', how='left')
test_functions.test_inner_join(tw_activity_df, week_tester[['w/c', 'WeekNumber_finYear']],
                               ['w/c'],
                               f"9_{platformID}_combination",
                               focus='left')



Inner join test 9_TWI_combination successful: No issues found.
...updating logbook...



In [6]:
tw_country_df = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country.csv",
                            low_memory=False)

tw_country_df['Channel ID'] = tw_country_df['Channel ID'].fillna('').apply(lambda x: str(int(x)))
tw_country_df = tw_country_df.drop_duplicates().drop(columns=['Week Commencing', 'Actual Week'])

tw_country_df['TW Linked FB account'] = tw_country_df['TW Linked FB account'].apply(
    lambda x: str(int(x)) if pd.notnull(x) and str(x).strip() != '' else ''
)

# TODO
# test uniqueness - account / week 
tw_country_df.sample()

,Channel ID,TW Account Name,TW Account Handle,Weekly Engagements,Weekly Video Views,WeekNumber_finYear,TW Linked FB account,TW Studios Exc UK,TWI_CountryName,Engagement %,TW Service Code,PlaceID
378125,63396727,Baby Cow,babycowLtd,27,0,35,,No,Australia,0.131004,WOR,AUS


# combine 

In [7]:
tw_activity_country = tw_activity_df.merge(tw_country_df, on=['Channel ID', 'WeekNumber_finYear'], 
                                           how='left', indicator=True)

test_functions.test_inner_join(tw_activity_df, tw_country_df, ['Channel ID', 'WeekNumber_finYear'], 
                               f"10_{platformID}_combination", 
                               test_step='joining activity & country - first')


Inner join test 10_TWI_combination failed: Issues found.
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...



In [8]:
left_over = tw_activity_country[tw_activity_country._merge == 'left_only'].drop(columns='_merge')
final_1 = tw_activity_country[tw_activity_country._merge != 'left_only'].drop(columns='_merge')

# groupby 
grouped_df = tw_country_df.groupby(['Channel ID', 'TW Account Name', 'TW Account Handle', 
                                    'TW Service Code', 'TW Linked FB account', 
                                    'TW Studios Exc UK', 'PlaceID']).agg({
    'Weekly Engagements': 'mean',
    'Engagement %': 'mean',
    'Weekly Video Views': 'mean'
}).reset_index()

# twitter_activity_metadata.columns: to keep the columns from the initial merge and loose all those that are empty anyway
final_2 = left_over[tw_activity_df.columns].merge(grouped_df, on='Channel ID', how='left')#.drop(columns='_merge')
test_functions.test_inner_join(left_over, grouped_df, ['Channel ID'], 
                               f"11_{platformID}_combination",
                               test_step='joining activity & country - second')


Inner join test 11_TWI_combination failed: Issues found.
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...



In [9]:
cols = ['Channel ID', 'TW Account Name', 'TW Account Handle',
        'Weekly Engagements', 'Weekly Video Views',
        'w/c', 'WeekNumber_finYear', 'TW Linked FB account', 'TW Studios Exc UK', 
        'PlaceID', 'Engagement %', 'TW Service Code' ]
final_df = pd.concat([final_1, final_2])[cols]

final_df['Weekly Engagements'] = np.where(final_df['Weekly Engagements']<0, 0, 
                                          final_df['Weekly Engagements'])

# TODO: find out why there are so many duplicates
print(final_df.shape)
print(final_df.drop_duplicates().shape)

final_df = final_df.rename(columns={'TW Service Code': 'ServiceID', }).drop_duplicates()
# file is used to compare to minnie's dataset:
final_df.to_csv(f"../data/processed/{platformID}/temp_{gam_info['file_timeinfo']}_metric_country.csv", index=None)


(6329, 12)
(6329, 12)


In [10]:
# handle if country == 'other'
regular_country = final_df[final_df['PlaceID'] != 'UNK']
regular_country_100 = final_df[final_df['PlaceID'] != 'UNK']
regular_country_100 = regular_country_100.groupby(['Channel ID', 'WeekNumber_finYear'])['Engagement %'].sum().reset_index()

rescaled_df = regular_country.merge(regular_country_100, on=['Channel ID', 'WeekNumber_finYear'], how='left',
                                    suffixes=['_', 'newTotal'])

rescaled_df['engagement_%'] = rescaled_df["Engagement %_"]/rescaled_df["Engagement %newTotal"]

## facebook factor 

In [11]:
fb_factor = pd.read_excel("../helper/FB Factor for IG and TW.xlsx").drop_duplicates()
fb_factor['FB Page ID'] = fb_factor['FB Page ID'].apply(lambda x: str(int(x)))

fb_factor = fb_factor.rename(columns={'FB Service Code': 'ServiceID',
                                      'FB Page ID': 'TW Linked FB account',
                                      'Week Number': 'WeekNumber_finYear'})
twitter_df = rescaled_df.merge(fb_factor[['TW Linked FB account', 'WeekNumber_finYear', 'Factor']], 
                               on=['TW Linked FB account', 'WeekNumber_finYear'], 
                               how='left', indicator=True)
#print(f"1: {twitter_df.columns}")
print(f"1: {twitter_df.shape}")

done = twitter_df[twitter_df['_merge'] == 'both'].drop(columns=['_merge'])
need_fix = twitter_df[twitter_df['_merge'] == 'left_only'].drop(columns=['_merge', 'Factor'])

fb_factor_service = fb_factor.groupby(['ServiceID', 'WeekNumber_finYear'])['Factor'].mean().reset_index()
fixed = need_fix.merge(fb_factor_service, on=['ServiceID', 'WeekNumber_finYear'], how='left')
twitter_df = pd.concat([done, fixed])
#print(f"2: {twitter_df.columns}")
print(f"2: {twitter_df.shape}")

fb_videoMetric = pd.read_excel("../helper/FB Video Metrics.xlsx")
fb_videoMetric = fb_videoMetric.rename(columns={
    'FB Page ID': 'TW Linked FB account',
    'GAM Week Number': 'WeekNumber_finYear',
    'FB Service Code': 'ServiceID'
})

fb_videoMetric['TW Linked FB account'] = fb_videoMetric['TW Linked FB account'].apply(
    lambda x: str(int(x)) if pd.notnull(x) and str(x).strip() != '' else ''
)

twitter_df = twitter_df.merge(fb_videoMetric, 
                              on=['TW Linked FB account', 'WeekNumber_finYear', 'ServiceID'], 
                              how='left', indicator=True)
#print(f"3: {twitter_df.columns}")
print(f"3: {twitter_df.shape}")

done = twitter_df[twitter_df['_merge'] == 'both'].drop(columns=['_merge'])
need_fix = twitter_df[twitter_df['_merge'] == 'left_only'].drop(columns=['_merge'])\
        .drop(columns=['30 VTR%', '30 Sec Views per Viewer'])
unmatched_vtr = fb_videoMetric.groupby(['ServiceID', 'WeekNumber_finYear'])[['30 VTR%', '30 Sec Views per Viewer']].mean().reset_index()
fixed = need_fix.merge(unmatched_vtr, on=['ServiceID', 'WeekNumber_finYear'], how='left')

twitter_df = pd.concat([done, fixed])
#print(f"4: {fixed.columns}")
print(f"4: {twitter_df.shape}")

1: (7099, 16)
2: (7099, 15)
3: (7099, 21)
4: (7099, 20)


In [12]:
twitter_df['Weekly Video Views'] = twitter_df['Weekly Video Views'].fillna(0)
twitter_df['30 sec viewers'] = twitter_df['30 VTR%']*twitter_df['Weekly Video Views']/(twitter_df['30 Sec Views per Viewer']*1.1)
twitter_df['temp'] = twitter_df['Weekly Engagements']/twitter_df['Factor']
twitter_df['Twitter Engaged Users'] = np.where(twitter_df['temp']>twitter_df['30 sec viewers'], 
                                                  twitter_df['temp']+twitter_df['30 sec viewers']*0.0733,
                                                    twitter_df['30 sec viewers']+twitter_df['temp']*0.2042)

twitter_df['uv_by_country'] = twitter_df["Twitter Engaged Users"]*twitter_df["engagement_%"]


In [13]:

cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country']
twitter_df[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer_country.csv",
                        index=None)